In [ ]:
import json, pickle, os
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss

os.makedirs("indexes", exist_ok=True)
print("Imports OK ✓")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK ✓


In [ ]:
with open("data/corpus_functions.json") as f:
    all_functions = json.load(f)

corpus = [f["source"] for f in all_functions]
print(f"Corpus loaded: {len(corpus)} functions")

Corpus loaded: 5902 functions


In [ ]:

print("Chunking strategy: function-level ✓")

Chunking strategy: function-level ✓


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding corpus (this may take 2-5 minutes)...")
embeddings = model.encode(
    corpus,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64
)

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(index, "indexes/faiss_index.bin")
np.save("indexes/embeddings.npy", embeddings)

print(f"FAISS index built: {index.ntotal} vectors ✓")

Loading weights: 100%|█████████████████████████████████████████████████████| 103/103 [00:00<00:00, 13723.86it/s]


Encoding corpus (this may take 2-5 minutes)...


Batches: 100%|██████████████████████████████████████████████████████████████████| 93/93 [00:32<00:00,  2.83it/s]

FAISS index built: 5902 vectors ✓


In [ ]:
tokenized_corpus = [doc.split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

with open("indexes/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

print("BM25 index saved ✓")

BM25 index saved ✓


In [ ]:
with open("indexes/corpus.json", "w") as f:
    json.dump(corpus, f)

print(f"Corpus saved: {len(corpus)} entries ✓")
print("\nAll indexes ready in /indexes:")
for fname in os.listdir("indexes"):
    size = os.path.getsize(f"indexes/{fname}") / 1024
    print(f"  {fname}: {size:.1f} KB")

Corpus saved: 5902 entries ✓

All indexes ready in /indexes:
  corpus.json: 3245.4 KB
  bm25.pkl: 3784.1 KB
  faiss_index.bin: 8853.0 KB
  embeddings.npy: 8853.1 KB
